# Hydraulic System Component Failure Prediction - Machine Learning Pipeline

This notebook implements an end-to-end classification pipeline to predict hydraulic system failure (`failure_within_50_hours`) using sensor telemetry and operational data.

**Workflow Architecture:**
1. **Data Ingestion & Comprehensive EDA** (Shape, Column Types, Missingness, Target Distribution, Summary Stats)
2. **Feature Preprocessing & Scaling** (`SimpleImputer` + `StandardScaler` + `OneHotEncoder` via `ColumnTransformer`)
3. **Stratified Train/Test Split** (80/20 with reproducible seed `random_state=42`)
4. **Class-Balanced Random Forest Training** (Optimized hyperparameters)
5. **Feature Importance Ranking** (Sorted diagnostic table)
6. **Example Prediction** (Predicted class & probability score)
7. **Pipeline Serialization** (Saved to `src/ML/model/hydraulic_system_failure_model.pkl`)
8. **Final Evaluation Matrix & Diagnostics** (Confusion matrix counts, normalized matrix, extended diagnostic metrics, and full classification report)

## 1. Setup & Library Imports

In [12]:
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 2. Load Dataset (`hydraulic_system.csv`)

In [13]:
try:
    df = pd.read_csv("../Data/hydraulic_system.csv")
except Exception:
    try:
        df = pd.read_csv("Data/hydraulic_system.csv")
    except Exception:
        try:
            df = pd.read_csv("src/ML/Data/hydraulic_system.csv")
        except Exception:
            df = pd.read_csv(
                "C:/Users/Jevil/OneDrive/Desktop/bob/bob-ai-hackathon-NexGen/src/ML/Data/hydraulic_system.csv"
            )

print("Hydraulic System dataset successfully loaded into pandas DataFrame.")

Hydraulic System dataset successfully loaded into pandas DataFrame.


## 3. Comprehensive Dataset Overview
Displaying dataset shape, column names, data types, missing value audit, class distribution, and numerical statistics.

In [14]:
print("=" * 60)
print("DATASET SHAPE:")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print("=" * 60)

print("\nCOLUMN NAMES:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

print("\n" + "=" * 60)
print("DATA TYPES:")
print(df.dtypes)

print("\n" + "=" * 60)
print("MISSING VALUES:")
missing = df.isnull().sum()
print(missing)
print(f"\nTotal Missing Values across dataset: {missing.sum()}")

print("\n" + "=" * 60)
print("CLASS DISTRIBUTION OF TARGET (failure_within_50_hours):")
target_counts = df["failure_within_50_hours"].value_counts()
target_pct = df["failure_within_50_hours"].value_counts(normalize=True) * 100
dist_df = pd.DataFrame({"Count": target_counts, "Percentage (%)": target_pct.round(2)})
dist_df.index = ["Normal (0)", "Failure within 50h (1)"]
print(dist_df)

print("\n" + "=" * 60)
print("SUMMARY STATISTICS FOR NUMERICAL FEATURES:")
print(df.describe().T[["mean", "std", "min", "25%", "50%", "75%", "max"]].round(2))

DATASET SHAPE:
Rows: 4889, Columns: 18

COLUMN NAMES:
   1. asset_id
   2. timestamp
   3. component_id
   4. component_type
   5. temperature
   6. vibration
   7. oil_pressure
   8. fuel_pressure
   9. rpm
  10. hydraulic_pressure
  11. battery_voltage
  12. coolant_temperature
  13. operating_hours
  14. load_percentage
  15. ambient_temperature
  16. sensor_status
  17. anomaly_label
  18. failure_within_50_hours

DATA TYPES:
asset_id                       str
timestamp                      str
component_id                   str
component_type                 str
temperature                float64
vibration                  float64
oil_pressure               float64
fuel_pressure              float64
rpm                        float64
hydraulic_pressure         float64
battery_voltage            float64
coolant_temperature        float64
operating_hours              int64
load_percentage            float64
ambient_temperature        float64
sensor_status                  str
anomal

## 4. Preprocessing Pipeline & Feature Scaling
- Exclude non-feature identifiers: `asset_id`, `timestamp`, `component_id`, `component_type`, and target `failure_within_50_hours`.
- Numerical pipeline: `SimpleImputer(strategy="median")` followed by `StandardScaler()` for zero-mean, unit-variance feature scaling.
- Categorical pipeline: `SimpleImputer(strategy="most_frequent")` followed by `OneHotEncoder(handle_unknown="ignore")` for `sensor_status`.
- Assembled in a single unified `ColumnTransformer`.

In [15]:
target = "failure_within_50_hours"
drop_cols = ["asset_id", "timestamp", "component_id", "component_type", target,"anomaly_label"]

X = df.drop(columns=drop_cols)
y = df[target]

num_cols = X.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()

print(f"Selected Numerical Features ({len(num_cols)}):\n{num_cols}\n")
print(f"Selected Categorical Features ({len(cat_cols)}):\n{cat_cols}\n")

num_transformer = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

cat_transformer = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_transformer, num_cols),
        ("cat", cat_transformer, cat_cols),
    ],
    verbose_feature_names_out=False,
)
print("Preprocessing pipeline with feature scaling configured.")

Selected Numerical Features (11):
['temperature', 'vibration', 'oil_pressure', 'fuel_pressure', 'rpm', 'hydraulic_pressure', 'battery_voltage', 'coolant_temperature', 'operating_hours', 'load_percentage', 'ambient_temperature']

Selected Categorical Features (1):
['sensor_status']

Preprocessing pipeline with feature scaling configured.


## 5. Stratified Train-Test Split (80/20)
Splitting data with `stratify=y` to preserve exact failure class proportions across train and test sets.

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("=" * 60)
print("DATASET SPLIT SUMMARY:")
print(f"  Total samples:  {len(df)}")
print(f"  X_train samples: {len(X_train)} ({len(X_train)/len(df)*100:.1f}%)")
print(f"  X_test samples:  {len(X_test)} ({len(X_test)/len(df)*100:.1f}%)")
print(f"  y_train samples: {len(y_train)} ({len(y_train)/len(df)*100:.1f}%)")
print(f"  y_test samples:  {len(y_test)} ({len(y_test)/len(df)*100:.1f}%)")
print("=" * 60)

DATASET SPLIT SUMMARY:
  Total samples:  4889
  X_train samples: 3911 (80.0%)
  X_test samples:  978 (20.0%)
  y_train samples: 3911 (80.0%)
  y_test samples:  978 (20.0%)


## 6. Model Training with Optimized Random Forest Classifier
Constructing an end-to-end `Pipeline` incorporating both the preprocessor with scaling and the `RandomForestClassifier` with balanced weighting to optimize accuracy and F1 score.

In [17]:
pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=16,
                min_samples_split=4,
                min_samples_leaf=2,
                class_weight="balanced_subsample",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

pipeline.fit(X_train, y_train)

final_feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()
print("Model training complete.")
print(f"\nFinal feature names used by the model ({len(final_feature_names)}):\n{list(final_feature_names)}")

Model training complete.

Final feature names used by the model (13):
['temperature', 'vibration', 'oil_pressure', 'fuel_pressure', 'rpm', 'hydraulic_pressure', 'battery_voltage', 'coolant_temperature', 'operating_hours', 'load_percentage', 'ambient_temperature', 'sensor_status_Normal', 'sensor_status_Warning']


## 7. Feature Importance Analysis
Ranking all scaled sensor and operational features by Gini importance from highest to lowest.

In [18]:
importances = pipeline.named_steps["classifier"].feature_importances_
fi_df = pd.DataFrame(
    {"Feature": final_feature_names, "Importance": importances}
).sort_values(by="Importance", ascending=False).reset_index(drop=True)

print("=" * 60)
print("FEATURE IMPORTANCES (Highest to Lowest):")
print("=" * 60)
print(fi_df.to_string(index=False))

FEATURE IMPORTANCES (Highest to Lowest):
              Feature  Importance
          temperature    0.124122
            vibration    0.115778
      load_percentage    0.087274
  coolant_temperature    0.084376
         oil_pressure    0.083609
      battery_voltage    0.070634
      operating_hours    0.069325
   hydraulic_pressure    0.067538
        fuel_pressure    0.064634
  ambient_temperature    0.063577
                  rpm    0.062361
 sensor_status_Normal    0.054566
sensor_status_Warning    0.052206


## 8. Model Evaluation on Training and Testing Sets

In [19]:
y_train_pred = pipeline.predict(X_train)
y_test_pred = pipeline.predict(X_test)
y_test_prob = pipeline.predict_proba(X_test)[:, 1]

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)
prec = precision_score(y_test, y_test_pred)
rec = recall_score(y_test, y_test_pred)
f1 = f1_score(y_test, y_test_pred)
roc_auc = roc_auc_score(y_test, y_test_prob)

print("=" * 60)
print("MODEL EVALUATION METRICS SUMMARY:")
print("=" * 60)
print(f"  Training Accuracy:  {train_acc:.4f} ({train_acc*100:.2f}%)")
print(f"  Testing Accuracy:   {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"  Precision:          {prec:.4f} ({prec*100:.2f}%)")
print(f"  Recall:             {rec:.4f} ({rec*100:.2f}%)")
print(f"  F1 Score:           {f1:.4f}")
print(f"  ROC-AUC Score:      {roc_auc:.4f}")
print("=" * 60)

MODEL EVALUATION METRICS SUMMARY:
  Training Accuracy:  0.9895 (98.95%)
  Testing Accuracy:   0.7485 (74.85%)
  Precision:          0.6277 (62.77%)
  Recall:             0.5443 (54.43%)
  F1 Score:           0.5831
  ROC-AUC Score:      0.7768


## 9. Example Prediction on Raw Telemetry Reading

In [20]:
example_sample = X_test.iloc[[0]]
example_pred = pipeline.predict(example_sample)[0]
example_prob = pipeline.predict_proba(example_sample)[0]

print("=" * 60)
print("EXAMPLE PREDICTION ON RAW SENSOR INPUT:")
print("=" * 60)
print("Input Sample Values:")
for k, v in example_sample.to_dict(orient="records")[0].items():
    print(f"  {k:22s}: {v}")
print("-" * 60)
print(f"Predicted Class: {example_pred} ('{'Failure' if example_pred == 1 else 'Normal'}')")
print(f"Normal Probability:  {example_prob[0]:.4f} ({example_prob[0]*100:.2f}%)")
print(f"Failure Probability: {example_prob[1]:.4f} ({example_prob[1]*100:.2f}%)")
print("=" * 60)

EXAMPLE PREDICTION ON RAW SENSOR INPUT:
Input Sample Values:
  temperature           : 77.459
  vibration             : 3.785
  oil_pressure          : 74.742
  fuel_pressure         : 52.539
  rpm                   : 1761.275
  hydraulic_pressure    : 149.804
  battery_voltage       : 23.026
  coolant_temperature   : 80.841
  operating_hours       : 643
  load_percentage       : 66.147
  ambient_temperature   : 31.074
  sensor_status         : Normal
------------------------------------------------------------
Predicted Class: 0 ('Normal')
Normal Probability:  0.6055 (60.55%)
Failure Probability: 0.3945 (39.45%)


## 10. Save Complete Pipeline to `src/ML/model/hydraulic_system_failure_model.pkl`

In [21]:
model_dir = None
for candidate in [
    Path("../model"),
    Path("model"),
    Path("src/ML/model"),
    Path("C:/Users/Jevil/OneDrive/Desktop/bob/bob-ai-hackathon-NexGen/src/ML/model"),
]:
    if candidate.exists():
        model_dir = candidate
        break
if model_dir is None:
    model_dir = Path("C:/Users/Jevil/OneDrive/Desktop/bob/bob-ai-hackathon-NexGen/src/ML/model")
    model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "hydraulic_system_failure_model.pkl"

with open(model_path, "wb") as f:
    pickle.dump(pipeline, f)

print(f"Model and preprocessing pipeline saved successfully to:\n  {model_path}")

with open(model_path, "rb") as f:
    loaded_pipeline = pickle.load(f)

test_pred = loaded_pipeline.predict(example_sample)[0]
test_prob = loaded_pipeline.predict_proba(example_sample)[0][1]
print("\nLoaded Model Verification from Pickle:")
print(f"  Verified Predicted Class:      {test_pred}")
print(f"  Verified Failure Probability:  {test_prob:.4f}")

Model and preprocessing pipeline saved successfully to:
  ..\model\hydraulic_system_failure_model.pkl

Loaded Model Verification from Pickle:
  Verified Predicted Class:      0
  Verified Failure Probability:  0.3945


## 11. Evaluation Matrix & Diagnostic Report (At the End)
Comprehensive evaluation matrix displaying the sample count confusion matrix, normalized confusion matrix, diagnostic counts, extended performance statistics, and full classification report.

In [22]:
cm = confusion_matrix(y_test, y_test_pred)
tn, fp, fn, tp = cm.ravel()

cm_df = pd.DataFrame(
    cm,
    index=["Actual Normal (0)", "Actual Failure (1)"],
    columns=["Predicted Normal (0)", "Predicted Failure (1)"],
)

cm_norm = confusion_matrix(y_test, y_test_pred, normalize="true") * 100
cm_norm_df = pd.DataFrame(
    cm_norm.round(2),
    index=["Actual Normal (0)", "Actual Failure (1)"],
    columns=["Predicted Normal (%)", "Predicted Failure (%)"],
)

specificity = tn / (tn + fp)
npv = tn / (tn + fn)
mcc = ((tp * tn) - (fp * fn)) / np.sqrt(float((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)))

print("=" * 70)
print("                     FINAL EVALUATION MATRIX")
print("=" * 70)

print("\n1. CONFUSION MATRIX (Sample Counts):")
print("-" * 50)
print(cm_df)

print("\n2. CONFUSION MATRIX (Normalized Class-wise %):")
print("-" * 50)
print(cm_norm_df)

print("\n3. MATRIX BREAKDOWN:")
print("-" * 50)
print(f"  True Negatives  (TN): {tn:>5}  (Correctly identified normal hydraulic systems)")
print(f"  False Positives (FP): {fp:>5}  (False alarms - normal flagged as failure)")
print(f"  False Negatives (FN): {fn:>5}  (Missed failures - failure flagged as normal)")
print(f"  True Positives  (TP): {tp:>5}  (Correctly identified failures)")

print("\n4. EXTENDED DIAGNOSTIC METRICS:")
print("-" * 50)
print(f"  Accuracy:             {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"  Sensitivity (Recall): {rec:.4f} ({rec*100:.2f}%)")
print(f"  Specificity:          {specificity:.4f} ({specificity*100:.2f}%)")
print(f"  Precision (PPV):      {prec:.4f} ({prec*100:.2f}%)")
print(f"  Negative Pred Value:  {npv:.4f} ({npv*100:.2f}%)")
print(f"  F1 Score:             {f1:.4f}")
print(f"  ROC AUC Score:        {roc_auc:.4f}")
print(f"  Matthews Corr Coef:   {mcc:.4f}")

print("\n5. FULL CLASSIFICATION REPORT:")
print("-" * 70)
print(classification_report(y_test, y_test_pred, target_names=["Normal (0)", "Failure (1)"], digits=4))
print("=" * 70)

                     FINAL EVALUATION MATRIX

1. CONFUSION MATRIX (Sample Counts):
--------------------------------------------------
                    Predicted Normal (0)  Predicted Failure (1)
Actual Normal (0)                    560                    102
Actual Failure (1)                   144                    172

2. CONFUSION MATRIX (Normalized Class-wise %):
--------------------------------------------------
                    Predicted Normal (%)  Predicted Failure (%)
Actual Normal (0)                  84.59                  15.41
Actual Failure (1)                 45.57                  54.43

3. MATRIX BREAKDOWN:
--------------------------------------------------
  True Negatives  (TN):   560  (Correctly identified normal hydraulic systems)
  False Positives (FP):   102  (False alarms - normal flagged as failure)
  False Negatives (FN):   144  (Missed failures - failure flagged as normal)
  True Positives  (TP):   172  (Correctly identified failures)

4. EXTENDED DIAG

## 12. Inference on Test Dataset (`hydraulic_system_test.csv`)
Load the unseen hydraulic system test data (`hydraulic_system_test.csv`), extract feature columns, generate failure predictions with an operational risk threshold (classified as failure `1` if probability exceeds 40%, otherwise `0`), format the predicted columns, and save the predictions back to disk.

In [13]:
# 1. Locate and load hydraulic_system_test.csv
test_data_path = None
for candidate in [
    Path("../Data/hydraulic_system_test.csv"),
    Path("Data/hydraulic_system_test.csv"),
    Path("src/ML/Data/hydraulic_system_test.csv"),
    Path("C:/Users/Jevil/OneDrive/Desktop/bob/bob-ai-hackathon-NexGen/src/ML/Data/hydraulic_system_test.csv"),
]:
    if candidate.exists():
        test_data_path = candidate
        break

if test_data_path is None:
    raise FileNotFoundError("Could not find hydraulic_system_test.csv in expected data paths.")

df_test = pd.read_csv(test_data_path)
print("=" * 70)
print(f"Loaded Test Dataset from: {test_data_path}")
print(f"Initial Shape: {df_test.shape[0]} rows, {df_test.shape[1]} columns")
print("=" * 70)

# 2. Extract feature columns (keeping existing preprocessing and feature selection)
drop_test_cols = ["asset_id", "timestamp", "component_id", "component_type", "failure_within_50_hours", "failure_probability_percent"]
X_test_input = df_test.drop(columns=[c for c in drop_test_cols if c in df_test.columns])

print("\nFeatures passed into pipeline for prediction:")
print(X_test_input.columns.tolist())

# 3. Predict failure probability using the trained pipeline
failure_probability = pipeline.predict_proba(X_test_input)[:, 1] * 100

# 4. Apply 40% threshold: if failure probability is above 40%, flag as failure (1), else normal (0)
df_test["failure_probability_percent"] = failure_probability.round(2)
df_test["failure_within_50_hours"] = (df_test["failure_probability_percent"] > 40.0).astype(int)

# 5. Save the updated file back to disk
df_test.to_csv(test_data_path, index=False)
print(f"\n[SUCCESS] Successfully written {len(df_test)} rows to '{test_data_path}'")
print(f"Final Shape: {df_test.shape[0]} rows, {df_test.shape[1]} columns")

# 6. Verifications
assert set(df_test["failure_within_50_hours"].unique()).issubset({0, 1}), "Invalid values in failure_within_50_hours"
assert (df_test["failure_probability_percent"] >= 0).all() and (df_test["failure_probability_percent"] <= 100).all(), "Invalid values in failure_probability_percent"
assert df_test.isnull().sum().sum() == 0, "Missing/null values detected"
assert len(df_test) == 978, f"Expected 978 rows, got {len(df_test)}"

print("\n" + "=" * 70)
print("TEST SET PREDICTION & VERIFICATION SUMMARY (Threshold: > 40% -> Failure)")
print("=" * 70)
counts = df_test["failure_within_50_hours"].value_counts()
pcts = df_test["failure_within_50_hours"].value_counts(normalize=True) * 100
summary_table = pd.DataFrame({"Count": counts, "Percentage (%)": pcts.round(2)})
summary_table.index = ["Failure within 50h (1)" if idx == 1 else "Normal (0)" for idx in summary_table.index]
print(summary_table)
print(f"\n- Predicted Failures (>40%):        {sum(df_test['failure_within_50_hours'] == 1)} ({sum(df_test['failure_within_50_hours'] == 1)/len(df_test)*100:.2f}%)")
print(f"- Predicted Normal (<=40%):          {sum(df_test['failure_within_50_hours'] == 0)} ({sum(df_test['failure_within_50_hours'] == 0)/len(df_test)*100:.2f}%)")
print(f"- failure_probability_percent range: [{df_test['failure_probability_percent'].min():.2f}%, {df_test['failure_probability_percent'].max():.2f}%]")
print(f"- Mean failure probability:          {df_test['failure_probability_percent'].mean():.2f}%")

# 7. Display preview of required columns
preview_cols = [
    "asset_id",
    "timestamp",
    "component_id",
    "failure_within_50_hours",
    "failure_probability_percent",
]
print("\n" + "=" * 70)
print("REQUIRED COLUMNS PREVIEW (First 10 Rows):")
print("=" * 70)
print(df_test[preview_cols].head(10))


Loaded Test Dataset from: ..\Data\hydraulic_system_test.csv
Initial Shape: 978 rows, 17 columns

Features passed into pipeline for prediction:
['temperature', 'vibration', 'oil_pressure', 'fuel_pressure', 'rpm', 'hydraulic_pressure', 'battery_voltage', 'coolant_temperature', 'operating_hours', 'load_percentage', 'ambient_temperature', 'sensor_status']

[SUCCESS] Successfully written 978 rows to '..\Data\hydraulic_system_test.csv'
Final Shape: 978 rows, 18 columns

TEST SET PREDICTION & VERIFICATION SUMMARY (Threshold: > 40% -> Failure)
                        Count  Percentage (%)
Failure within 50h (1)    772           78.94
Normal (0)                206           21.06

- Predicted Failures (>40%):        772 (78.94%)
- Predicted Normal (<=40%):          206 (21.06%)
- failure_probability_percent range: [12.87%, 96.11%]
- Mean failure probability:          64.9%

REQUIRED COLUMNS PREVIEW (First 10 Rows):
  asset_id            timestamp component_id  failure_within_50_hours  failure_p